In [13]:
import importlib
import config

importlib.reload(config)

<module 'config' from '/home/sarthak/programs/Transformer/config.py'>

In [27]:
from pathlib import Path
import torch
import torch.nn as nn
from config import get_config, latest_weights_file_path
from train import get_model, get_ds, run_validation
from translate import translate
from train import greedy_decode

In [28]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)


Using device: cuda
Max length of source sentence: 434
Max length of target sentence: 288


In [29]:
model_filename = latest_weights_file_path(config)
print("Model path:", model_filename)

Model path: iitb_weights/tmodel09.pt


In [30]:
# Load the pretrained weights
model_filename = latest_weights_file_path(config)
state = torch.load(model_filename)
model.load_state_dict(state['model_state_dict'])

<All keys matched successfully>

In [31]:
run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, config['seq_len'], device, lambda msg: print(msg), 0, None, num_examples=10)

stty: Inappropriate ioctl for device


--------------------------------------------------------------------------------
    SOURCE: Interpreter
    TARGET: इंटरफेस
 PREDICTED: 
--------------------------------------------------------------------------------
    SOURCE: From my hotel in Berlin I was able to get a list of vegetarian restaurants in Europe and it was very useful to me in my tour.
    TARGET: बर्लिन में मैं एक ऐसे रेस्तरां में जाकर खया करता था जहां शाकाहार मिल सकता था। 
 PREDICTED: को बहुत अधिक लगा गया ।
--------------------------------------------------------------------------------
    SOURCE: The transformation must be integral, and integral therefore the rejection of all that withstands it.
    TARGET: रूपांतर होगा सर्वागीण, इसलिये जो कुछ उस रूपांतर में बाधक है उसका त्याग भी होना होगा सर्वागीण। 
 PREDICTED: इसमें यह है कि और हैं ।
--------------------------------------------------------------------------------
    SOURCE: It 's a potential, right?
    TARGET: यह एक संभावित, सही है? 
 PREDICTED: यह एक निश्चित

In [32]:
def translate_sentence(sentence, model, tokenizer_src, tokenizer_tgt, config, device):
    model.eval()

    src_tokens = tokenizer_src.encode(sentence).ids

    sos_id = tokenizer_src.token_to_id('[SOS]')
    eos_id = tokenizer_src.token_to_id('[EOS]')
    pad_id = tokenizer_src.token_to_id('[PAD]')

    src_tokens = [sos_id] + src_tokens + [eos_id]

    if len(src_tokens) < config['seq_len']:
        src_tokens += [pad_id] * (config['seq_len'] - len(src_tokens))
    else:
        src_tokens = src_tokens[:config['seq_len']]

    source = torch.tensor(src_tokens).unsqueeze(0).to(device)

    source_mask = (source != pad_id).unsqueeze(1).unsqueeze(2).int().to(device)

    output_tokens = greedy_decode(
        model, source, source_mask,
        tokenizer_src, tokenizer_tgt,
        config['seq_len'], device
    )
    translated = tokenizer_tgt.decode(output_tokens.detach().cpu().numpy())

    return translated

In [34]:
sentence = "Testing the trained model by sarthak"

output = translate_sentence(
    sentence,
    model,
    tokenizer_src,
    tokenizer_tgt,
    config,
    device
)

print("Input:", sentence)
print("Output:", output)

Input: Testing the trained model by sarthak
Output: सत्र द्वारा के लिए
